# Tournament walkthrough

Demonstrates the full tournament pipeline:

1. Write a tiny custom tournament config (1 RL agent vs 2 scripted bots,
   1 iteration, 1 map = 4 games).
2. Run it via the CLI.
3. Parse the produced CSV.
4. Plot the standings.

End-to-end: ~30-60 seconds on a CPU laptop. For the full 19-agent
thesis result, see `examples/showcase_results.ipynb` (parses the shipped
tournament CSV without running anything).

**Prereqs**: `bash setup/local.sh` once, then run this notebook from
the repo root.

## 1. Sanity check

In [ ]:
import shutil

from microrts_agent.paths import PROJECT_ROOT

agent_dir = PROJECT_ROOT / "data" / "agents" / "UECD-SingleMap-Best"
bridge_jar = PROJECT_ROOT / "microrts_agent" / "microrts" / "lib" / "bridge.jar"
assert agent_dir.exists(), f"FAIL: {agent_dir} missing. Run bash setup/local.sh"
assert bridge_jar.exists(), (
    "FAIL: bridge.jar missing. Run bash microrts_agent/microrts/build_bridge.sh"
)
assert shutil.which("java"), "FAIL: Java not on PATH (need JDK 17+)"
print("Prereqs OK")
print("Repo root :", PROJECT_ROOT)

## 2. Write a tiny tournament config

The full shipped config (`microrts_agent/tournament_configs/single_map.json`)
runs 19 agents over 5 iterations = 1710 games. For the walkthrough, we
subset to 1 RL agent + 2 scripted bots and a single iteration.

The RL agent is referenced with `agent:<path-to-its-dir>` (relative to
the config file's location). Scripted bots are referenced by their
`AI_MAPPING` key from `microrts_agent/registries/ai.py`.

In [ ]:
import json

config = {
    "maps": ["maps/open_competition/basesWorkers16x16A.xml"],
    "ais": [
        f"agent:{agent_dir}",  # UECD-SingleMap-Best
        "WorkerRush",
        "CoacAI",
    ],
    "iterations": 1,
    "maxGameLengths": [4000],
    "timeBudget": 100,
    "iterationsBudget": -1,
    "preAnalysisBudget": 3_600_000,
    "fullObservability": True,
    "selfMatches": False,
    "timeoutCheck": False,
    "runGC": False,
    "saveTraces": False,
    "saveGameLogs": False,
    "slowAIs": [],
}

config_dir = PROJECT_ROOT / "outputs" / "tournament_walkthrough"
config_dir.mkdir(parents=True, exist_ok=True)
config_path = config_dir / "mini.json"
with open(config_path, "w") as f:
    json.dump(config, f, indent=2)

print("Config written to:", config_path)
print("AIs:", config["ais"])
print(
    "Games expected: 3 AIs x 2 others x 1 map x 1 iter x 2 positions = 6 (self-matches off, no duplicates)"
)

## 3. Run the mini tournament

`microrts-agent tournament run --config <path>` reads the JSON above,
spawns the games, and writes the result CSV under
`outputs/tournaments/<config-name>/`.

In [ ]:
import subprocess
import sys

cmd = [
    sys.executable,
    "-m",
    "microrts_agent",
    "tournament",
    "run",
    str(config_path),
]
print("Running:", " ".join(cmd))
result = subprocess.run(cmd, cwd=str(PROJECT_ROOT), capture_output=True, text=True, timeout=600)
print(result.stdout[-2000:])
if result.returncode != 0:
    print("--- stderr ---")
    print(result.stderr[-2000:])

## 4. Parse the produced CSV

The runner writes a structured CSV at `outputs/tournaments/<name>/tournament.csv`.
The `tournament parse` subcommand converts it to a more workable JSON
(one record per game). Same parser the shipped data was built with.

In [ ]:
out_dir = PROJECT_ROOT / "outputs" / "tournaments" / "mini"
parse_cmd = [
    sys.executable,
    "-m",
    "microrts_agent",
    "tournament",
    "parse",
    str(out_dir),
]
result = subprocess.run(
    parse_cmd, cwd=str(PROJECT_ROOT), capture_output=True, text=True, timeout=60
)
print(result.stdout[-1000:])

parsed_path = out_dir / "tournament_parsed.json"
with open(parsed_path) as f:
    data = json.load(f)
print(f"\nGames recorded: {len(data['games'])}")
for g in data["games"]:
    p = g["players"]
    r = g["result"]
    winner = (
        p["ai1"]["name"] if r["winner"] == 0 else p["ai2"]["name"] if r["winner"] == 1 else "draw"
    )
    print(f"  {p['ai1']['name']:30s} vs {p['ai2']['name']:30s} -> {winner} ({r['time']} steps)")

## 5. Plot the standings

Same head-to-head matrix logic as `examples/showcase_results.ipynb`,
but on the 3-AI subset.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

rows = []
for g in data["games"]:
    a1, a2 = g["players"]["ai1"]["name"], g["players"]["ai2"]["name"]
    w = g["result"]["winner"]
    rows.append(
        {"agent": a1, "opponent": a2, "won": 1 if w == 0 else 0, "drew": 1 if w == -1 else 0}
    )
    rows.append(
        {"agent": a2, "opponent": a1, "won": 1 if w == 1 else 0, "drew": 1 if w == -1 else 0}
    )

df = pd.DataFrame(rows)
agg = (
    df.groupby(["agent", "opponent"])
    .agg(wins=("won", "sum"), draws=("drew", "sum"), games=("won", "count"))
    .reset_index()
)
agg["score"] = (agg["wins"] + 0.5 * agg["draws"]) / agg["games"]

ais = sorted(df["agent"].unique())
matrix = agg.pivot(index="agent", columns="opponent", values="score").reindex(
    index=ais, columns=ais
)

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(matrix.values * 100, cmap="RdYlGn", vmin=0, vmax=100)
ax.set_xticks(range(len(ais)))
ax.set_yticks(range(len(ais)))
ax.set_xticklabels(ais, rotation=30, ha="right")
ax.set_yticklabels(ais)
for i in range(len(ais)):
    for j in range(len(ais)):
        v = matrix.values[i, j]
        if pd.notna(v):
            ax.text(j, i, f"{v * 100:.0f}", ha="center", va="center", fontsize=11)
ax.set_title("Mini tournament: score matrix (row vs col)")
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()

## Next steps

- Reproduce the full thesis result: `microrts-agent tournament run
  single_map` (uses `microrts_agent/tournament_configs/single_map.json`,
  takes hours but matches `data/tournaments/single_map/` to within run
  variance).
- Visualise existing tournament results: `microrts-agent tournament viz
  outputs/tournaments/<name>/`. Produces the PDF tree under
  `visualizations/{basic-metrics,game-theoretic-metrics}/` that the
  shipped tournaments mirror.
- Add more agents to the mini config above: just append their
  `agent:<path>` strings to `config['ais']` and re-run.